# Downsampling Eaglei Data and Merging with County and ERA5 Data

The goal of this notebook is to:
- Downsample the eaglei data for each year to a 6-hour cadence, replacing the number of customers without power with the 6-hour mean
- Combine data for all years into a single file
- Export the combined data
- Merge additional county info
- Export the merged data

What I don't have working yet:
- Look up info from the ERA5 dataset about temperature, precipitation, snow depth, and wind speed

(I'm getting stuck on merging the weatherbench xarray with the eaglei data)

In [2]:
import pandas as pd
import numpy as np
import os

Downsample the eaglei data

The data appear to only exist when the number of customers without power is nonzero. So we'll first need to upsample to a true 15-minute cadence and fill in missing values with 0.
(Note that this is a bit of an assumption that only non-zero values are being reported, rather than values being missing.)

Then we'll downsample to a 6-hour cadence by computing the mean number of customers out over the 6-hour window.

Note that this code chunk takes roughly 5 minutes to run.

In [2]:
#Get a list of files that start with eaglei_outages_ in the directory ../Data/eaglei_data/
files = os.listdir('../Data/eaglei_data/')
files = [f for f in files if f.startswith('eaglei_outages_2')]

#Create an empty data frame
outages = pd.DataFrame()

for filename in files:
    #Open the file
    df = pd.read_csv('../Data/eaglei_data/' + filename)

    #Drop the county and state variables to facilitate up/downsampling (these could be merged back in later if needed)
    df.drop(['county', 'state'], axis=1, inplace=True)

    #Convert the run_start_time variable into datetime
    df['run_start_time'] = pd.to_datetime(df['run_start_time'])
    df.set_index('run_start_time', inplace=True)

    #Group the data by fips_code. Add new values of run_start_time every 15 minutes.
    # Forward fill values of customers_out with 0
    # Then ungroup the data
    df = df.groupby('fips_code').resample('15min').asfreq().fillna(0).reset_index(level=0, drop=True)

    #In the fips_code variable, replace 0 with NA
    df['fips_code'] = df['fips_code'].replace(0, np.nan)

    #Forward fill the fips_code with the most recent value
    df['fips_code'] = df['fips_code'].ffill()

    #Group the data by fips_code. Then downsample the data to every 6 hours, replacing customers_out with the mean and then ungroup the data
    df = df.groupby('fips_code').resample('6h').mean().reset_index(level=0, drop=True)


    #Concatenate df with outages
    outages = pd.concat([outages, df])

#For exporting and future merging, move the datetime back to a "regular" variable and reset the index
outages['datetime'] = pd.to_datetime(outages.index)
outages.reset_index(drop=True, inplace=True)


In [21]:
#Export outages to a parquet
outages.to_parquet('../Data/eaglei_data/eaglei_outages.parquet')

In [2]:
#Or, if the parquet file has already been saved, load the parquet
outages = pd.read_parquet('../Data/eaglei_data/eaglei_outages.parquet')

Next, we can merge the outages data with the additional county variables from the Counties_All file

In [3]:
#Add a YEAR variable from datetime
outages['YEAR'] = outages['datetime'].dt.year

#Load ../Data/Counties_All.csv
counties = pd.read_csv('../Data/Counties_All.csv')

# Merge outages and counties based on the YEAR variable and the fips_code/FIPS variables
outages_merged = outages.merge(counties, left_on=['YEAR', 'fips_code'], right_on=['YEAR', 'FIPS'])

In [3]:
#Export outages_merged to a parquet
outages_merged.to_parquet('../Data/eaglei_data/eaglei_outages_with_county_info.parquet')

NameError: name 'outages_merged' is not defined

In [4]:
#Or, if the parquet file has already been saved, load the parquet
outages_merged = pd.read_parquet('../Data/eaglei_data/eaglei_outages_with_county_info.parquet')

## Loading the ERA5 Data

We will open the Zarr data with XArray after getting GCS permissions. We can test bucket access with fsspec:

In [5]:
import fsspec

fs = fsspec.filesystem('gs')
fs.ls('gs://weatherbench2/datasets/era5/')

['weatherbench2/datasets/era5/',
 'weatherbench2/datasets/era5/1959-2022-1h-240x121_equiangular_with_poles_conservative.zarr',
 'weatherbench2/datasets/era5/1959-2022-1h-360x181_equiangular_with_poles_conservative.zarr',
 'weatherbench2/datasets/era5/1959-2022-6h-128x64_equiangular_conservative.zarr',
 'weatherbench2/datasets/era5/1959-2022-6h-128x64_equiangular_with_poles_conservative.zarr',
 'weatherbench2/datasets/era5/1959-2022-6h-1440x721.zarr',
 'weatherbench2/datasets/era5/1959-2022-6h-240x121_equiangular_with_poles_conservative.zarr',
 'weatherbench2/datasets/era5/1959-2022-6h-512x256_equiangular_conservative.zarr',
 'weatherbench2/datasets/era5/1959-2022-6h-64x32_equiangular_conservative.zarr',
 'weatherbench2/datasets/era5/1959-2022-6h-64x32_equiangular_with_poles_conservative.zarr',
 'weatherbench2/datasets/era5/1959-2022-6h-64x33.zarr',
 'weatherbench2/datasets/era5/1959-2022-full_37-1h-0p25deg-chunk-1.zarr-v2',
 'weatherbench2/datasets/era5/1959-2022-full_37-6h-0p25deg-chu

Next, we'll load the 6-hour downsampled data set

In [6]:
import xarray as xr

reanalysis = xr.open_zarr(
    'gs://weatherbench2/datasets/era5/1959-2023_01_10-wb13-6h-1440x721_with_derived_variables.zarr', 
    chunks={'time': 48},
    consolidated=True,
    decode_timedelta=True    
)

Let's reduce the size by only including data since 1/1/2014 and only our target variables:
- 10m_wind_speed
- 2m_temperature
- snow_depth
- total_precipitation_12hr
- total_precipitation_24hr
- total_precipitation_6hr
- wind_speed

In [7]:
#Select time values since 1/1/2014, latitudes/longitudes in the continental US, and the following variables: 
features = [
    'latitude', 
    'longitude', 
    'time', 
    '10m_wind_speed', 
    '2m_temperature', 
    'snow_depth', 
    'total_precipitation_12hr', 
    'total_precipitation_24hr', 
    'total_precipitation_6hr', 
]

reanalysis = reanalysis.sel(time=slice('2014', '2021'), latitude=slice(50, 24), longitude=slice(235, 294))[features]

In [ ]:
#ERA5 uses a [0, 360] domain for longitude rather than degrees Easty/West. So we'll add 360 to all values of centroid_longitude
outages_merged['centroid_longitude'] = outages_merged['centroid_longitude'] + 360

Next we'll try to merge the reanalysis and outages data. We'll just start with data from January 2015 to cut down on runtime and memory

In [ ]:
#Slice reanalysis to just january 2015
reanalysis_jan_2015 = reanalysis.sel(time=slice('2015-01-01', '2015-01-31'))

#Make a subset of outages_merged for just January, 2015
outages_jan_2015 = outages_merged.loc[(outages_merged['datetime'] >= '2015-01-01') & (outages_merged['datetime'] < '2015-02-01')]

#In outages_jan_2015, rename centroid_latitude as latitude, centroid_longitude as longitude, and datetime as time
outages_jan_2015.rename(columns={'centroid_latitude': 'latitude', 'centroid_longitude': 'longitude', 'datetime': 'time'}, inplace=True)

#Create a multiindex for outages_jan_2015 based on datetime, centroid_latitude, and centroid_longitude
outages_jan_2015.set_index(['time', 'latitude', 'longitude'], inplace=True)

#Remove duplicate values from the multiindex
outages_jan_2015 = outages_jan_2015[~outages_jan_2015.index.duplicated(keep='first')]

/var/folders/gs/h69vjsl512b8200d29lxpgrc0000gn/T/ipykernel_36626/1545251759.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  outages_jan_2015.rename(columns={'centroid_latitude': 'latitude', 'centroid_longitude': 'longitude', 'datetime': 'time'}, inplace=True)


Next we'll convert outages_jan_2015 into an xarray to facilitate merging.

For some reason, my data values are all replaced with NaN after this conversion and I can't figure out why. So I'm stuck!

In [ ]:
#Convert outages_jan_2015 into an xarray
outages_jan_2015 = outages_jan_2015.to_xarray()

ValueError: cannot convert a DataFrame with a non-unique MultiIndex into xarray

In [41]:
outages_jan_2015

<xarray.Dataset> Size: 267MB
Dimensions:                           (time: 124, latitude: 93, longitude: 181)
Coordinates:
  * time                              (time) datetime64[ns] 992B 2015-01-01 ....
  * latitude                          (latitude) float64 744B 25.25 ... 48.5
  * longitude                         (longitude) float64 1kB 236.5 ... 292.2
Data variables: (12/16)
    fips_code                         (time, latitude, longitude) float64 17MB ...
    customers_out                     (time, latitude, longitude) float64 17MB ...
    YEAR                              (time, latitude, longitude) float64 17MB ...
    NAME                              (time, latitude, longitude) object 17MB ...
    STUSPS                            (time, latitude, longitude) object 17MB ...
    FIPS                              (time, latitude, longitude) float64 17MB ...
    ...                                ...
    POPULATION                        (time, latitude, longitude) float64 17MB ...
    BUILDVALUE                        (time, latitude, longitude) float64 17MB ...
    AGRIVALUE                         (time, latitude, longitude) float64 17MB ...
    AREA                              (time, latitude, longitude) float64 17MB ...
    SOVI_SCORE                        (time, latitude, longitude) float64 17MB ...
    Power_Dependent_Devices_DME_Mean  (time, latitude, longitude) float64 17MB ...

In [21]:
#Merge outages_jan_2015 with reanalysis_jan_2015 using the datetime, centroid_latitude, and centroid_longitude coordintes from outages_jan_2015 and the time, latitude, and longitude coordinates from reanalysis_jan_2015
merged = xr.merge([outages_jan_2015, reanalysis_jan_2015])
